# Notebook 30 v29.2 — 360° Feature-Space EDA with Leakage-Aware Outlier Treatment

Includes before/after five-bin distributions, training-only winsorization, L1/L2/Ridge recalibration, and all original stock/sector diagnostics.


In [ ]:
from pathlib import Path
import sys, os, warnings
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks':ROOT=ROOT.parent
if not (ROOT/'src').exists():raise FileNotFoundError('Open notebook inside stock_ml_dynamic repository')
sys.path.insert(0,str(ROOT/'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.classifier_benchmark import eligible_universe
from stock_ml_dynamic.feature_ablation import development_holdout_split
from stock_ml_dynamic.historical_market import add_historical_market_features
from stock_ml_dynamic.accuracy_features import add_accuracy_features
from stock_ml_dynamic.volume_regime_research import add_volume_regime_features
from stock_ml_dynamic.feature_space_eda import *
plt.rcParams['figure.figsize']=(11,5);sns.set_theme(style='whitegrid')
SEED=42;MAX_EDA_ROWS=80000;CORR_ROWS=80000;REG_ROWS=60000;MIN_OVERLAP=80


## 1. Load and audit the development universe
The existing labeled master returns four objects. We explicitly exclude the final historical holdout **before engineering and descriptive summaries**. We also report the original master size and development cutoff.

In [ ]:
paths=[ROOT/'artifacts/master_tables/1d',ROOT/'notebooks/artifacts/master_tables/1d']
source=next((p for p in paths if (p/'master_labeled.parquet').exists()),None)
if source is None:raise FileNotFoundError('Cannot locate artifacts/master_tables/1d/master_labeled.parquet')
master,latest,bundle,manifest=load_master(source)
eligible,excluded=eligible_universe(master)
dev,cutoff=development_holdout_split(eligible,.2)
print('Master rows:',len(master),'Eligible rows:',len(eligible),'Development rows:',len(dev),'Development cutoff:',cutoff)
print('Symbols:',dev.symbol.nunique(),'Intervals:',dev.interval_code.value_counts().to_dict())
print('Development timestamp range:',dev.bar_timestamp.min(),dev.bar_timestamp.max())
print('Excluded groups:',len(excluded) if hasattr(excluded,'__len__') else excluded)
display(dev[['symbol','interval_code','bar_timestamp','direction','next_return']].head())


## 2. Engineer the same historical, technical, and volume features used in the RF research
Feature computation is performed **only** on the development subset. Raw OHLCV fields are retained for diagnostics, but are not silently added to the model feature inventory.

In [ ]:
expanded,market_cols=add_historical_market_features(dev)
expanded,technical_cols=add_accuracy_features(expanded)
expanded,volume_cols=add_volume_regime_features(expanded)
model_features=list(dict.fromkeys(market_cols+technical_cols+volume_cols))
num,cat=feature_inventory(expanded,model_features)
raw_num=[c for c in ('open_price','high_price','low_price','close_price','adj_close','volume','next_return') if c in expanded]
raw_cat=[c for c in ('symbol','interval_code','direction') if c in expanded]
print('Engineered numerical features:',len(num),'categorical engineered:',len(cat))
print('Raw numerical diagnostics:',raw_num,'raw categorical diagnostics:',raw_cat)
print('Duplicate ticker-timestamp rows:',expanded.duplicated(['symbol','interval_code','bar_timestamp']).sum())


## 2A. Robust outlier treatment — training-only bounds and before/after EDA

**Important:** Outliers are not automatically erroneous. This notebook uses *winsorization* (clipping feature values at training-derived quantiles), **not row deletion**. Dropping rare observations can destroy useful market-stress signals and distort class prevalence. Raw prices, labels, next returns, stock/sector return correlations, and the historical holdout are **never changed**. Bounds are fitted only on the earlier 75% chronological training portion of the development set; the later 25% is a diagnostic validation window. Zero-heavy indicators are not shifted or log-transformed; low-variation columns are skipped. This is exploratory preprocessing, **not** an independently validated trading model.


In [ ]:
from stock_ml_dynamic.outlier_eda import fit_outlier_bounds, apply_outlier_bounds, compare_numeric_profiles
raw_expanded=expanded.copy()
train,test,eda_cutoff=time_split(raw_expanded,train_frac=.75)
print('Train-only clipping fit cutoff:',eda_cutoff,'train:',int(train.sum()),'validation:',int(test.sum()))
# Adjust these two tail quantiles to explore sensitivity; do not select them using validation results.
CLIP_LOWER=.005;CLIP_UPPER=.995
bounds=fit_outlier_bounds(raw_expanded,num,train,lower=CLIP_LOWER,upper=CLIP_UPPER)
expanded,clip_report=apply_outlier_bounds(raw_expanded,bounds)
profile_comparison=compare_numeric_profiles(raw_expanded,expanded,num)
print('Clipping applies to engineered numerical predictors only. Original OHLCV and targets remain unchanged.')
display(clip_report.sort_values('clipped_pct',ascending=False).head(30))
display(profile_comparison.reindex(clip_report.sort_values('clipped_pct',ascending=False).index).head(30))
assert expanded[['direction','next_return','bar_timestamp','symbol']].equals(raw_expanded[['direction','next_return','bar_timestamp','symbol']])


In [ ]:
# Show both five-bin raw and five-bin clipped distributions; no arbitrary x-axis truncation.
plot_cols=clip_report.sort_values('clipped_pct',ascending=False).head(24).index.tolist()
for start in range(0,len(plot_cols),6):
    chunk=plot_cols[start:start+6]
    fig,axes=plt.subplots(len(chunk),2,figsize=(15,max(3,len(chunk)*2.6)),squeeze=False)
    for i,col in enumerate(chunk):
        for j,(name,data) in enumerate((('Raw',raw_expanded),('Winsorized',expanded))):
            x=pd.to_numeric(data[col],errors='coerce').replace([np.inf,-np.inf],np.nan).dropna()
            if len(x)>MAX_EDA_ROWS:x=x.sample(MAX_EDA_ROWS,random_state=SEED)
            axes[i,j].hist(x,bins=5,edgecolor='white')
            axes[i,j].set_title(f'{col} — {name} (5 bins)',fontsize=10)
    plt.tight_layout();plt.show()


## 2B. Recalibrate L1/L2 and Ridge after outlier treatment — same chronological split

This comparison reruns exactly the same regularization settings and train/validation dates with and without clipping. Scaling and imputation remain training-only inside the estimator pipeline. Differences are descriptive; neither clipping quantiles nor models should be selected using this validation set and then reported as independently validated performance. The subsequent full EDA sections use the clipped **predictor** matrix; stock/sector correlations continue using the untouched raw returns.


In [ ]:
raw_metrics,raw_coef=regularization_paths(raw_expanded,num,train,test,max_rows=REG_ROWS,seed=SEED)
clean_metrics,clean_coef=regularization_paths(expanded,num,train,test,max_rows=REG_ROWS,seed=SEED)
raw_metrics=raw_metrics.assign(preprocessing='raw')
clean_metrics=clean_metrics.assign(preprocessing='winsorized')
recalibration=pd.concat([raw_metrics,clean_metrics],ignore_index=True)
print('L1/L2 classification and Ridge return regression — before vs after clipping:')
display(recalibration.sort_values(['model','preprocessing','C'] if 'C' in recalibration else ['model','preprocessing']))
for metric in ('accuracy','auc','r2','mae'):
    if metric in recalibration:
        pivot=recalibration.pivot_table(index=['model','C'] if metric in ('accuracy','auc') else ['model','alpha'],columns='preprocessing',values=metric)
        if not pivot.empty:
            pivot['winsorized_minus_raw']=pivot.get('winsorized',np.nan)-pivot.get('raw',np.nan)
            print(metric);display(pivot)


## 3. Categorical feature distributions — bar charts
Displays every engineered categorical variable and selected identifiers/labels. High-cardinality symbols are shown as the top 30 plus a separate full count table. No label is treated as a predictive input.

In [ ]:
for col in list(dict.fromkeys(raw_cat+cat)):
    counts=expanded[col].fillna('MISSING').astype(str).value_counts()
    display(pd.DataFrame({'category':counts.index,'count':counts.values,'share':counts.values/len(expanded)}).head(40))
    ax=counts.head(30).plot.bar(figsize=(max(9,min(17,len(counts)*.32)),4),title=f'{col} — counts (top 30)')
    ax.set_ylabel('Observations');plt.xticks(rotation=65,ha='right');plt.tight_layout();plt.show()


## 4. Numerical feature distributions — **five-bin histograms for every numeric variable**
A deterministic sample caps plotting cost. Histograms show **five equal-width bins** per variable; extreme outliers may compress the middle bins, so quantiles and robust-range plots follow. This includes raw OHLCV and target returns (clearly marked as diagnostics).

In [ ]:
sample=expanded.sample(n=min(MAX_EDA_ROWS,len(expanded)),random_state=SEED)
all_numeric=list(dict.fromkeys(raw_num+num))
for start in range(0,len(all_numeric),12):
    chunk=all_numeric[start:start+12]
    fig,axes=plt.subplots(3,4,figsize=(18,10));axes=axes.ravel()
    for ax,col in zip(axes,chunk):
        v=pd.to_numeric(sample[col],errors='coerce').replace([np.inf,-np.inf],np.nan).dropna()
        if len(v):ax.hist(v,bins=5,edgecolor='white');ax.set_title(f'{col} (n={len(v):,})',fontsize=9)
        else:ax.set_title(f'{col} — no finite values',fontsize=9)
    for ax in axes[len(chunk):]:ax.axis('off')
    plt.suptitle('Exactly 5 bins per numeric variable',fontsize=14);plt.tight_layout();plt.show()


## 5. Completeness, outliers, skew, and feature stability
Review missingness, zeros, skewness, extreme quantiles, and possible constant features before modeling. The time-bucketed missingness panel can expose changes in data coverage.

In [ ]:
profile=numeric_profile(expanded,all_numeric)
display(profile)
fig,axes=plt.subplots(1,2,figsize=(17,6))
profile.missing_pct.sort_values(ascending=False).head(25).plot.bar(ax=axes[0],title='Top missingness rates (%)')
profile['skew'].abs().sort_values(ascending=False).head(25).plot.bar(ax=axes[1],title='Largest absolute skewness')
plt.tight_layout();plt.show()
print('Constant features:',profile.index[profile['unique']<=1].tolist())


In [ ]:
q=sample[num].replace([np.inf,-np.inf],np.nan).quantile([.01,.25,.5,.75,.99]).T
q['iqr']=q[.75]-q[.25];q['tail_span_iqr']=(q[.99]-q[.01])/q['iqr'].replace(0,np.nan)
display(q.sort_values('tail_span_iqr',ascending=False).head(30))


## 6. Numerical correlation matrix and redundant-feature clusters
Pearson and Spearman matrices use pairwise available observations. This is an **exploratory sample**, not training-time feature selection. Also inspect highest absolute correlations and a hierarchical correlation clustermap.

In [ ]:
corr_frame=sample[num].replace([np.inf,-np.inf],np.nan)
pearson=corr_frame.corr(method='pearson',min_periods=100)
spearman=corr_frame.corr(method='spearman',min_periods=100)
for label,mat in [('Pearson',pearson),('Spearman',spearman)]:
    fig,ax=plt.subplots(figsize=(max(12,len(num)*.24),max(10,len(num)*.24)))
    sns.heatmap(mat,cmap='coolwarm',center=0,vmin=-1,vmax=1,xticklabels=True,yticklabels=True,ax=ax)
    ax.set_title(f'{label} numerical-feature correlation matrix');plt.tight_layout();plt.show()
print('Strongest Pearson pairs:');display(correlation_pairs(pearson,40))
print('Strongest Spearman pairs:');display(correlation_pairs(spearman,40))


In [ ]:
if len(num)>2:
    clustered=pearson.fillna(0)
    sns.clustermap(clustered,cmap='coolwarm',center=0,vmin=-1,vmax=1,figsize=(max(12,len(num)*.23),max(12,len(num)*.23)))
    plt.show()


## 7. Target relationship diagnostics
For every numeric predictor, report univariate correlations with next-bar return, class-conditional medians, and information about missingness by class. **Do not** include target-derived columns among predictors.

In [ ]:
target=expanded[['next_return','direction']].copy()
feature_target=expanded[num].replace([np.inf,-np.inf],np.nan)
assoc=feature_target.corrwith(pd.to_numeric(target.next_return,errors='coerce')).rename('pearson_next_return').to_frame()
assoc['spearman_next_return']=feature_target.corrwith(pd.to_numeric(target.next_return,errors='coerce'),method='spearman')
assoc['abs_pearson']=assoc.pearson_next_return.abs()
display(assoc.sort_values('abs_pearson',ascending=False).head(40))
display(expanded.groupby('direction')[num].median().T)


## 8. L1 and L2 regularized logistic regressions + Ridge regression
L1 (lasso-style) and L2 regularization are compared for the **three-class direction target**, with standardized coefficients, sparsity, ROC-AUC, and accuracy. Ridge regression separately predicts the **continuous next return**. Imputation/scaling are fit on training rows only. The chronological split excludes label overlap, and uses only the historical development window. Results are exploratory; do not treat this validation slice as the untouched final holdout.

In [ ]:
# Reuse the fixed chronological split defined before clipping
print('Chronological EDA cutoff:',eda_cutoff,'train rows:',train.sum(),'test rows:',test.sum())
metrics,coef=clean_metrics.drop(columns="preprocessing"),clean_coef
display(metrics.sort_values(['model','C'] if 'C' in metrics else ['model']))
fig,axes=plt.subplots(1,2,figsize=(15,5))
for model,g in metrics[metrics.model.str.startswith('logistic')].groupby('model'):
    axes[0].plot(g.C,g.accuracy,marker='o',label=model)
    axes[1].plot(g.C,g.nonzero,marker='o',label=model)
for ax in axes:ax.set_xscale('log');ax.legend();ax.set_xlabel('C (inverse regularization strength)')
axes[0].set_title('Chronological validation accuracy');axes[1].set_title('Number of nonzero class coefficients')
plt.tight_layout();plt.show()


In [ ]:
for penalty in ('l1','l2'):
    p=coef[(coef.model==penalty)&(coef.cls==1)].copy()
    for c in sorted(p.C.unique()):
        top=p[p.C==c].assign(magnitude=lambda d:d.coefficient.abs()).nlargest(12,'magnitude')
        display(top[['feature','coefficient']].assign(penalty=penalty,C=c))
    wide=p.pivot_table(index='feature',columns='C',values='coefficient')
    keep=wide.abs().max(axis=1).nlargest(20).index
    wide.loc[keep].T.plot(figsize=(12,6),title=f'{penalty.upper()} coefficient paths — class 1 (top 20)')
    plt.xscale('log');plt.axhline(0,color='gray',lw=.8);plt.tight_layout();plt.show()


## 9. Sector metadata discovery (read-only, explicit provenance)
We inspect **actual PostgreSQL column names** rather than assuming a schema. Prefer `mart.sp500_gics_reference` if it contains a sector and symbol. If sector data is absent or the DB is unavailable, sector analysis is skipped with an explicit warning. The reference is **not guaranteed point-in-time**: use sector correlations for descriptive analysis, not historical sector feature training without dated memberships.

In [ ]:
from sqlalchemy import inspect, text
from stock_ml_dynamic.db import engine
sectors=None;sector_source='unavailable'
try:
    with engine().connect() as conn:
        inspector=inspect(conn)
        for schema,table in [('mart','sp500_gics_reference'),('mart','symbols'),('staging','company_metadata')]:
            if not inspector.has_table(table,schema=schema):continue
            names=[x['name'] for x in inspector.get_columns(table,schema=schema)]
            print(f'{schema}.{table} columns:',names)
            sym=next((c for c in ('symbol','ticker') if c in names),None)
            sec=next((c for c in ('gics_sector','sector_name','sector','sector_label') if c in names),None)
            if sym and sec:
                # Identifiers are from inspected metadata, not arbitrary input.
                lookup=pd.read_sql(text(f'SELECT "{sym}" AS symbol, "{sec}" AS sector FROM "{schema}"."{table}" WHERE "{sec}" IS NOT NULL'),conn)
                lookup['symbol']=lookup.symbol.astype(str).str.upper().str.replace('.','-',regex=False)
                lookup['sector']=lookup.sector.astype(str)
                lookup=lookup.drop_duplicates('symbol')
                if len(lookup):sectors=lookup;sector_source=f'{schema}.{table}';break
except Exception as exc:
    print('Sector lookup unavailable:',type(exc).__name__,str(exc)[:250])
print('Sector source:',sector_source)
if sectors is not None:
    print('Matched tickers:',expanded.symbol.isin(sectors.symbol).sum(),'/',len(expanded))
    display(sectors.sector.value_counts().rename('tickers').to_frame())
else:print('No valid sector mapping found; sector plots will be skipped.')


## 10. Stock-to-stock returns correlation — full universe
Correlations use aligned **same-date adjusted-close returns**, not price levels. Missing dates are never forward-filled. Pairwise overlap must be at least 80 sessions; low-overlap pairs remain missing. Heatmap clustering makes co-movement visible.

In [ ]:
wide=aligned_returns(dev,MIN_OVERLAP)
print('Return matrix:',wide.shape,'date range:',wide.index.min(),wide.index.max())
stock_corr=wide.corr(min_periods=MIN_OVERLAP)
print('Strongest stock return correlations:');display(correlation_pairs(stock_corr,35))
if wide.shape[1]>1:
    sns.clustermap(stock_corr.fillna(0),cmap='coolwarm',center=0,vmin=-1,vmax=1,figsize=(15,15),xticklabels=False,yticklabels=False)
    plt.show()


## 11. Sector-to-sector correlations, within-sector stock correlations, and dispersion
Sector proxies are same-date **equal-weight averages of available constituents** (not investable sector ETFs). We display the sector correlation matrix, within-sector correlation distributions, highest/lowest pairs, and intra-sector return dispersion. Changing constituent coverage and current-sector classifications are material caveats.

In [ ]:
if sectors is None:
    print('SKIPPED: no sector mapping; check database connection or reference table.')
else:
    sector_returns=sector_return_series(wide,sectors,min_constituents=2)
    print('Sector proxy shape:',sector_returns.shape)
    if sector_returns.shape[1]>1:
        sector_corr=sector_returns.corr(min_periods=MIN_OVERLAP)
        plt.figure(figsize=(12,9));sns.heatmap(sector_corr,annot=True,fmt='.2f',cmap='coolwarm',center=0,vmin=-1,vmax=1)
        plt.title('Sector-to-sector daily return correlation');plt.tight_layout();plt.show()
        display(correlation_pairs(sector_corr,30))
    pairs=within_sector_summary(wide,sectors,MIN_OVERLAP)
    print('Valid within-sector pairs:',len(pairs))
    if not pairs.empty:
        display(pairs.groupby('sector').r.agg(['count','mean','median','std','min','max']).sort_values('mean',ascending=False))
        plt.figure(figsize=(14,6));sns.boxplot(data=pairs,x='sector',y='r');plt.xticks(rotation=65,ha='right');plt.title('Within-sector pairwise return correlations');plt.tight_layout();plt.show()
        display(pairs.nlargest(20,'r'));display(pairs.nsmallest(20,'r'))
    membership=sectors.set_index('symbol').sector
    dispersion={}
    for sec in membership.unique():
        symbols=[s for s in membership.index[membership==sec] if s in wide]
        if len(symbols)>1:dispersion[sec]=wide[symbols].std(axis=1).mean()
    display(pd.Series(dispersion,name='mean_daily_cross_sectional_std').sort_values(ascending=False).to_frame())


## 12. PCA and dimensionality, multicollinearity, and effective rank
PCA is descriptive, with imputation/scaling fitted on the sampled development rows. The explained-variance curve estimates how redundant the feature space is; loading plots identify which variables dominate the first components. This is **not** an inference-time transform.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
x=sample[num].replace([np.inf,-np.inf],np.nan)
x=SimpleImputer(strategy='median',keep_empty_features=True).fit_transform(x)
x=StandardScaler().fit_transform(x)
pca=PCA(n_components=min(x.shape[1],30),random_state=SEED).fit(x)
variance=np.cumsum(pca.explained_variance_ratio_)
fig,ax=plt.subplots(figsize=(11,5));ax.plot(np.arange(1,len(variance)+1),variance,marker='o')
ax.axhline(.90,ls='--',color='gray');ax.axhline(.95,ls=':',color='gray')
ax.set_xlabel('Components');ax.set_ylabel('Cumulative explained variance');ax.set_title('PCA feature-space effective dimensionality');plt.tight_layout();plt.show()
print('Components for 90%:',np.searchsorted(variance,.90)+1 if variance[-1]>=.90 else '>30')
loadings=pd.DataFrame(pca.components_[:min(5,len(pca.components_))].T,index=num,columns=[f'PC{i+1}' for i in range(min(5,len(pca.components_)))])
display(loadings)
plt.figure(figsize=(10,max(8,len(num)*.20)));sns.heatmap(loadings,cmap='coolwarm',center=0);plt.tight_layout();plt.show()


## 13. Temporal drift and ticker coverage
Track feature medians and missingness across time, along with ticker coverage and class proportions. Sudden shifts may reflect market regimes, data quality, corporate actions, or changing universe composition.

In [ ]:
d=expanded.copy();d['month']=pd.to_datetime(d.bar_timestamp,utc=True).dt.to_period('M').astype(str)
monthly=d.groupby('month').agg(rows=('symbol','size'),tickers=('symbol','nunique'),up_share=('direction',lambda x:(x==1).mean()))
display(monthly.tail(30))
monthly[['tickers','up_share']].plot(subplots=True,figsize=(13,7),title=['Ticker coverage','Class-1 prevalence']);plt.tight_layout();plt.show()
selected=assoc.abs_pearson.nlargest(min(8,len(assoc))).index.tolist()
if selected:
    medians=d.groupby('month')[selected].median()
    normalized=(medians-medians.mean())/medians.std().replace(0,np.nan)
    normalized.tail(36).plot(figsize=(14,6),title='Monthly median feature drift (z-scaled for display)');plt.tight_layout();plt.show()


## 14. Research synthesis and data-quality gate
Use the tables above to identify missing data, feature redundancy, sector clustering, unstable relationships, and regularization sensitivity. **No model or trading policy is promoted from this EDA.** The final historical holdout and prospective ledger were never loaded.

In [ ]:
print('FEATURE-SPACE AUDIT SUMMARY')
print('Engineered numerical features:',len(num))
print('Missingness > 10%:',profile.index[profile.missing_pct>10].tolist())
print('Near-duplicate feature pairs (|Pearson r| >= 0.95):',int((correlation_pairs(pearson,len(num)**2).abs_r>=.95).sum()))
print('Stock return matrix:',wide.shape)
print('Sector mapping:',sector_source)
print('L1/L2 best exploratory validation accuracy:')
display(metrics[metrics.model.str.startswith('logistic')].sort_values('accuracy',ascending=False).head(8))
print('Next: decide feature families to test under fresh nested chronological validation; do not choose a live strategy from EDA alone.')

print('Winsorized predictors:',int((clip_report.status=='clipped').sum()),'of',len(num))
print('Highest clipping rates:');display(clip_report.sort_values('clipped_pct',ascending=False).head(10))
